# Crop Recommendation System

This notebook focuses on recommending the best crop to plant based on chemical and physical properties of the soil and environment. The dataset includes:
- `N`: Ratio of Nitrogen content in soil
- `P`: Ratio of Phosphorus content in soil
- `K`: Ratio of Potassium content in soil
- `temperature`: Temperature in degrees Celsius
- `humidity`: Relative humidity in %
- `ph`: pH value of the soil
- `rainfall`: Rainfall in mm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import pickle
import warnings
warnings.filterwarnings('ignore')

### 1. Data Loading

In [ ]:
df = pd.read_csv('dataset/Crop_recommendation.csv')
df.head()

### 2. Data Exploration

In [ ]:
print("Shape of dataset:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())

In [ ]:
df['label'].unique()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.iloc[:, :-1].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Soil & Environmental Features')
plt.show()

### 3. Data Preprocessing

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}

model_accuracies = []
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    model_accuracies.append({'Model': name, 'Accuracy': acc})

acc_df = pd.DataFrame(model_accuracies).sort_values(by='Accuracy', ascending=False)
acc_df

### 5. Final Model Evaluation (Naive Bayes)

In [ ]:
final_model = GaussianNB()
final_model.fit(X_train, y_train)
preds = final_model.predict(X_test)

print("Final Model: Naive Bayes")
print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt='d', cmap='Greens', 
            xticklabels=final_model.classes_, yticklabels=final_model.classes_)
plt.title('Confusion Matrix - Naive Bayes')
plt.show()

### 6. Saving the Model

In [ ]:
import os
if not os.path.exists('models'):
    os.makedirs('models')
with open('models/crop_recommendation_nb.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print("Model saved as 'models/crop_recommendation_nb.pkl'")